# 13 — Nested Zones

## Goal
Detect when two zones **overlap significantly** and decide whether to merge them or treat them separately.  
A small zone nested inside a larger one provides a **sharper entry** (inner) with a **safer stop** (outer distal).

---

## Overlap Ratio

$$\text{overlap} = \max(0,\; \min(P_1, P_2) - \max(D_1, D_2))$$

$$\text{overlap\_ratio} = \frac{\text{overlap}}{\min(\text{width}_1,\; \text{width}_2)}$$

If $\text{overlap\_ratio} \geq 0.50$ → the zones are considered **nested** and merged.  
The merged zone keeps:
- **Proximal** = closer proximal (sharper entry)
- **Distal**   = farther distal  (safer stop)

---

## Merge rule

Only zones of the **same type** (both demand or both supply) are merged.  
Mixed types at the same price level indicate a contested area — keep them separate.

---

## Visualization
Original overlapping zones shown side-by-side; merged result shown as a wider box.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. `find_nested_pairs` and `merge_zones`

In [ ]:
OVERLAP_MIN = 0.50

def overlap_ratio(z1, z2):
    # both demand: proximal = top, distal = bottom
    # compare the [distal, proximal] intervals
    lo1, hi1 = min(z1["proximal"], z1["distal"]), max(z1["proximal"], z1["distal"])
    lo2, hi2 = min(z2["proximal"], z2["distal"]), max(z2["proximal"], z2["distal"])
    overlap   = max(0, min(hi1, hi2) - max(lo1, lo2))
    min_width = min(hi1 - lo1, hi2 - lo2)
    return overlap / min_width if min_width > 0 else 0

def find_nested_pairs(zones):
    pairs = []
    for i, z1 in enumerate(zones):
        for j, z2 in enumerate(zones):
            if j <= i: continue
            if z1["zone_type"] != z2["zone_type"]: continue
            if overlap_ratio(z1, z2) >= OVERLAP_MIN:
                pairs.append((i, j))
    return pairs

pairs = find_nested_pairs(zones)
print(f"Nested pairs found: {len(pairs)}")
for i, j in pairs:
    print(f"  Zone {i} ({zones[i]['scenario']}) ↔ Zone {j} ({zones[j]['scenario']})  "
          f"overlap_ratio={overlap_ratio(zones[i], zones[j]):.2f}")


In [ ]:
def merge_pair(z1, z2):
    # sharper entry = proximal closer to the price approach direction
    if z1["zone_type"] == "demand":
        merged_prox = min(z1["proximal"], z2["proximal"])   # lower top = sharper demand entry
        merged_dist = min(z1["distal"],   z2["distal"])     # lower bottom = safer stop
    else:
        merged_prox = max(z1["proximal"], z2["proximal"])
        merged_dist = max(z1["distal"],   z2["distal"])
    return {**z1,
            "proximal":   merged_prox,
            "distal":     merged_dist,
            "zone_width": abs(merged_prox - merged_dist),
            "scenario":   f"{z1['scenario']} ∪ {z2['scenario']}",
            "merged":     True}

merged_zones = list(zones)
merged_idxs  = set()
for i, j in pairs:
    m = merge_pair(zones[i], zones[j])
    merged_zones.append(m)
    merged_idxs.update([i, j])

final_zones = [z for k, z in enumerate(merged_zones)
               if k not in merged_idxs or k >= len(zones)]

print(f"Zones before merge: {len(zones)}  →  after: {len(final_zones)}")
for z in final_zones:
    tag = " [merged]" if z.get("merged") else ""
    print(f"  {z['zone_type']:8} {z['formation']}  {z.get('scenario','')}{tag}")


## 3. Visualization — nested vs merged zones

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

for z in final_zones:
    x0, x1  = df.index[z["bs"]], df.index[z["be"]]
    c_edge   = EDGE[z["zone_type"]]
    lw       = 2 if z.get("merged") else 1
    dash     = "solid" if z.get("merged") else "dot"
    alpha    = 0.20 if z.get("merged") else 0.08
    fill     = f"rgba(38,166,154,{alpha})" if z["zone_type"] == "demand" else f"rgba(239,83,80,{alpha})"
    fig.add_vrect(x0=x0, x1=x1, fillcolor=fill,
                  line=dict(color=c_edge, width=lw, dash=dash))
    fig.add_annotation(x=x0, y=z["proximal"],
                       text=f"{z['formation']}{'*' if z.get('merged') else ''}",
                       showarrow=False, font=dict(size=8, color=c_edge),
                       xanchor="left", yanchor="bottom")

fig.update_layout(
    title="Nested Zones — merged zones marked with *",
    height=520, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
